In [1]:
import genvarloader as gvl
import pooch
from pathlib import Path
import numpy as np
import src.vep_pipeline as vp
import src.clinvar as cv
import os,sys
print(sys.executable)
os.environ['PATH'] = '/grid/kinney/home/zhliu/.conda/envs/gvl_splicing/bin:' + os.environ['PATH']
!which python

/grid/kinney/home/zhliu/.conda/envs/gvl_splicing/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/grid/kinney/home/zhliu/.conda/envs/gvl_splicing/bin/python
~/.conda/envs/gvl_splicing/bin/python


In [2]:
os.getcwd()

'/grid/kinney/home/zhliu/vep_dna/VEP_splicing/testing'

In [3]:
# Download the correct 1KGP reference genome and haplotype variants
import os
import pooch
from src import config
from src.onekg import get_ftp_dict, list_remote_vcf, download_vcfs

# Set download directory
config.DATA_DIR = "/grid/kinney/home/zhliu/vep_dna/VEP_splicing/testing/data"

# Set 1000 Genomes key
key = "1000_Genomes_on_GRCh38"

# 1. Download the reference genome
reference_url = get_ftp_dict()[key]["ref"]
reference_path = pooch.retrieve(url=reference_url, known_hash=None)
print(f"Reference genome saved at: {reference_path}")

# Compress ref genome if not already
if not Path(f"{reference_path}.bgz").exists():
    !bgzip -c {reference_path} > {reference_path}.bgz

# 2. List all available VCFs
manifest = list_remote_vcf(key=key)

# 3. Download all VCFs (including .vcf.gz and .tbi index files)
vcf_files = download_vcfs(key=key, manifest=manifest, skip_checks=True, verbose=True)
print(f"Downloaded {len(vcf_files)} files")

# 4. Print resulting files
vcf_paths = {k: v for k, v in vcf_files.items() if not k.endswith("_idx")}
index_paths = {k: v for k, v in vcf_files.items() if k.endswith("_idx")}

print("VCF paths:")
for k, v in vcf_paths.items():
    print(f"{k}: {v}")

print("Index file paths:")
for k, v in index_paths.items():
    print(f"{k}: {v}")


Reference genome saved at: /grid/kinney/home/zhliu/.cache/pooch/296ca04ba1df562072adc4c76f64cfb9-GRCh38_full_analysis_set_plus_decoy_hla.fa


Skipping /grid/kinney/home/zhliu/vep_dna/VEP_splicing/testing/data/1000_Genomes_on_GRCh38/vcf/ALL.chr1.shapeit2_integrated_snvindels_v2a_27022019.GRCh38.phased.vcf.gz because it already exists
Skipping /grid/kinney/home/zhliu/vep_dna/VEP_splicing/testing/data/1000_Genomes_on_GRCh38/vcf/ALL.chr1.shapeit2_integrated_snvindels_v2a_27022019.GRCh38.phased.vcf.gz.tbi because it already exists
Skipping /grid/kinney/home/zhliu/vep_dna/VEP_splicing/testing/data/1000_Genomes_on_GRCh38/vcf/ALL.chr2.shapeit2_integrated_snvindels_v2a_27022019.GRCh38.phased.vcf.gz because it already exists
Skipping /grid/kinney/home/zhliu/vep_dna/VEP_splicing/testing/data/1000_Genomes_on_GRCh38/vcf/ALL.chr2.shapeit2_integrated_snvindels_v2a_27022019.GRCh38.phased.vcf.gz.tbi because it already exists
Skipping /grid/kinney/home/zhliu/vep_dna/VEP_splicing/testing/data/1000_Genomes_on_GRCh38/vcf/ALL.chr3.shapeit2_integrated_snvindels_v2a_27022019.GRCh38.phased.vcf.gz because it already exists
Skipping /grid/kinney/home/

In [7]:
# Rename, atomize, and index 1KGP haplotype variants for each chromosome
import subprocess
import pysam
from pathlib import Path

# Assume: vcf_paths is already defined and reference_bgz is the bgzipped reference FASTA
reference_bgz = Path(f"{reference_path}.bgz")
# Prepare rename file (if not already exists)
chr_rename_file = Path(config.DATA_DIR) / "1000_Genomes_on_GRCh38/chr_rename.txt"
if not chr_rename_file.exists():
    with open(chr_rename_file, "w") as f:
        for i in range(1, 23):
            f.write(f"{i}\tchr{i}\n")
        f.write("X\tchrX\nY\tchrY\nMT\tchrM\n")

for key, vcf_path in vcf_paths.items():
    chromosome = key.replace('_vcf', '')  # e.g., chr1
    vcf_path = Path(vcf_path)

    renamed_vcf = vcf_path.with_name(vcf_path.stem + ".renamed.vcf.gz")
    atomized_vcf = vcf_path.with_name(vcf_path.stem + ".atomized.vcf.gz")

    # Step 1: Rename chromosomes using bcftools view
    if not renamed_vcf.exists():
        print(f"Renaming chromosomes for: {vcf_path.name}")
        subprocess.run([
            "bcftools", "annotate",
            "--rename-chrs", str(chr_rename_file),
            "-Oz", "-o", str(renamed_vcf),
            str(vcf_path)
        ], check=True)
        
    # Step 2: Index renamed VCF
    if not Path(f"{renamed_vcf}.tbi").exists():
        print(f"Indexing renamed VCF: {renamed_vcf.name}")
        subprocess.run(["tabix", "-p", "vcf", str(renamed_vcf)], check=True)

    # Step 3: Atomize (normalize multi-allelics)
    if not atomized_vcf.exists():
        print(f"Atomizing: {renamed_vcf.name}")
        subprocess.run([
            "bcftools", "norm",
            "--threads", "16",  # <-- Adjust thread count as needed
            "-a", "-c", "s", "-m-any",
            "-f", reference_bgz,
            "-o", str(atomized_vcf),
            str(renamed_vcf)
        ], check=True)

    # Step 4: Index atomized VCF
    if not Path(f"{atomized_vcf}.tbi").exists():
        print(f"Indexing atomized VCF: {atomized_vcf.name}")
        subprocess.run(["tabix", "-p", "vcf", str(atomized_vcf)], check=True)

    # Optional: Verify with pysam
    vcf_in = pysam.VariantFile(str(atomized_vcf), "r")
    chroms = list(vcf_in.header.contigs)
    first_rec = next(vcf_in.fetch())
    print(f"✅ Chromosome renamed: {atomized_vcf.name} → Contigs: {chroms[:1]}, First record CHROM: {first_rec.chrom}")
    vcf_in.close()


✅ Chromosome renamed: ALL.chr1.shapeit2_integrated_snvindels_v2a_27022019.GRCh38.phased.vcf.atomized.vcf.gz → Contigs: ['chr1'], First record CHROM: chr1
✅ Chromosome renamed: ALL.chr2.shapeit2_integrated_snvindels_v2a_27022019.GRCh38.phased.vcf.atomized.vcf.gz → Contigs: ['chr2'], First record CHROM: chr2
✅ Chromosome renamed: ALL.chr3.shapeit2_integrated_snvindels_v2a_27022019.GRCh38.phased.vcf.atomized.vcf.gz → Contigs: ['chr3'], First record CHROM: chr3
✅ Chromosome renamed: ALL.chr4.shapeit2_integrated_snvindels_v2a_27022019.GRCh38.phased.vcf.atomized.vcf.gz → Contigs: ['chr4'], First record CHROM: chr4
✅ Chromosome renamed: ALL.chr5.shapeit2_integrated_snvindels_v2a_27022019.GRCh38.phased.vcf.atomized.vcf.gz → Contigs: ['chr5'], First record CHROM: chr5
✅ Chromosome renamed: ALL.chr6.shapeit2_integrated_snvindels_v2a_27022019.GRCh38.phased.vcf.atomized.vcf.gz → Contigs: ['chr6'], First record CHROM: chr6
✅ Chromosome renamed: ALL.chr7.shapeit2_integrated_snvindels_v2a_27022019.GR

In [4]:
import pysam

vcf = pysam.VariantFile("/grid/kinney/home/zhliu/vep_dna/VEP_splicing/testing/data/1000_Genomes_on_GRCh38/vcf/ALL.chr1.shapeit2_integrated_snvindels_v2a_27022019.GRCh38.phased.vcf.gz")

# Check header contigs
print("Header contigs:")
print(list(vcf.header.contigs))

# Check actual CHROM values in records
for rec in vcf.fetch():
    print("Example CHROM in record:", rec.contig)
    break


Header contigs:
['1']
Example CHROM in record: 1


In [43]:
# Split SpliceVarDB x ClinVar splicing SNVs VCF file based on chromosome, and create a bed region file for each split VCF file.
split_splicing_variants = False

if split_splicing_variants:
    import pandas as pd
    import subprocess
    import pooch
    
    filtered_intersect_snv = pd.read_csv('splicevardb_x_clinvar_snv.csv')
    
    # Remove rows with missing simplified clinical significance
    filtered_intersect_snv = filtered_intersect_snv[~filtered_intersect_snv['CLNSIG_simplified'].isna()]
    
    # Ensure 'chr' values are strings and start with 'chr' (e.g., 'chr1' instead of '1')
    filtered_intersect_snv['chr'] = filtered_intersect_snv['chr'].astype(str)
    
    reference = pooch.retrieve(
        url="https://ftp.1000genomes.ebi.ac.uk/vol1/ftp/technical/reference/GRCh38_reference_genome/GRCh38_full_analysis_set_plus_decoy_hla.fa",
        known_hash=None,
        progressbar=True,
    )
    reference = reference + ".bgz"
        
    unique_chrs = filtered_intersect_snv['chr'].unique()
    
    # Loop through chromosomes
    unique_chrs = filtered_intersect_snv['chr'].unique()
    
    for chromosome in unique_chrs:
        chr_df = filtered_intersect_snv[filtered_intersect_snv['chr'] == chromosome]
        base_name = f"splicevardb_x_clinvar_snv_{chromosome}"
        raw_vcf = f"./data/splicevardb_x_clinvar/{base_name}.vcf"
        compressed_vcf = f"{raw_vcf}.gz"
        atomized_vcf = f"./data/splicevardb_x_clinvar/{base_name}.atomized.vcf.gz"
    
        # Write minimal VCF
        with open(raw_vcf, "w") as f:
            f.write('##fileformat=VCFv4.1\n')
            f.write('##source=splicevardb.ipynb\n') 
            f.write('##reference=GRCh38\n')
            f.write('##ID=<Description="ClinVar Variation ID">\n')
            f.write(f"##contig=<ID={chromosome}>\n")
            f.write('#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO\n')
            for _, row in chr_df.iterrows():
                info_str = ""  # Empty INFO field
                f.write(f"{row['chr']}\t{row['pos']}\t{row['clinvar_id']}\t{row['ref']}\t{row['alt']}\t{row['qual']}\t{row['filter']}\t{info_str}\n")
    
        # Compress and index raw VCF
        print(f"Compressing {raw_vcf}")
        subprocess.run(["bgzip", "-k", raw_vcf], check=True)
    
        print(f"Indexing {compressed_vcf}")
        subprocess.run(["tabix", "-p", "vcf", compressed_vcf], check=True)
    
        # Normalize using reference
        print(f"Normalizing {compressed_vcf}")
        subprocess.run([
            "bcftools", "norm",
            "-a", "-c", "s", "-m-any",
            "-f", reference,
            "-o", atomized_vcf,
            compressed_vcf
        ], check=True)
    
        print(f"Indexing {atomized_vcf}")
        subprocess.run(["tabix", "-p", "vcf", atomized_vcf], check=True)

        vcf_data = pd.read_csv(atomized_vcf, 
                        sep='\t',
                        comment='#', # Skip comment lines starting with #
                        names=['CHROM', 'POS', 'ID', 'REF', 'ALT', 'QUAL', 'FILTER', 'INFO']) # Standard VCF columns


        site_bed = pd.DataFrame()
        site_bed['CHROM'] = vcf_data['CHROM']
        site_bed['start'] = vcf_data['POS']
        site_bed['end'] = vcf_data['POS'] + 1  # This is weird, ask Jack
        display(site_bed.head())
        padding = 15000
        site_bed.iloc[:,1] = site_bed.iloc[:,1]-padding
        site_bed.iloc[:,2] = site_bed.iloc[:,2]+padding
        # Save BED in same naming convention
        bed_output_path = f"./data/splicevardb_x_clinvar/{base_name}_region_bed.bed"
        site_bed.to_csv(bed_output_path, sep='\t', header=False, index=False)
        print(f"Saved padded BED to {bed_output_path}")

Compressing ./data/splicevardb_x_clinvar/splicevardb_x_clinvar_snv_chr1.vcf
Indexing ./data/splicevardb_x_clinvar/splicevardb_x_clinvar_snv_chr1.vcf.gz
Normalizing ./data/splicevardb_x_clinvar/splicevardb_x_clinvar_snv_chr1.vcf.gz
Indexing ./data/splicevardb_x_clinvar/splicevardb_x_clinvar_snv_chr1.atomized.vcf.gz


Lines   total/split/joined/realigned/removed/skipped:	560/0/0/0/0/0
REF/ALT total/modified/added:  	560/0/0


,CHROM,start,end
0,chr1,1049450,1049451
1,chr1,1051638,1051639
2,chr1,3244069,3244070
3,chr1,3244097,3244098
4,chr1,3244101,3244102


Saved padded BED to ./data/splicevardb_x_clinvar/splicevardb_x_clinvar_snv_chr1_region_bed.bed
Compressing ./data/splicevardb_x_clinvar/splicevardb_x_clinvar_snv_chr10.vcf
Indexing ./data/splicevardb_x_clinvar/splicevardb_x_clinvar_snv_chr10.vcf.gz
Normalizing ./data/splicevardb_x_clinvar/splicevardb_x_clinvar_snv_chr10.vcf.gz
Indexing ./data/splicevardb_x_clinvar/splicevardb_x_clinvar_snv_chr10.atomized.vcf.gz


Lines   total/split/joined/realigned/removed/skipped:	311/0/0/0/0/0
REF/ALT total/modified/added:  	311/0/0


...

,CHROM,start,end
0,chr9,332378,332379
1,chr9,332381,332382
2,chr9,332468,332469
3,chr9,332472,332473
4,chr9,386311,386312


Saved padded BED to ./data/splicevardb_x_clinvar/splicevardb_x_clinvar_snv_chr9_region_bed.bed
Compressing ./data/splicevardb_x_clinvar/splicevardb_x_clinvar_snv_chrX.vcf
Indexing ./data/splicevardb_x_clinvar/splicevardb_x_clinvar_snv_chrX.vcf.gz
Normalizing ./data/splicevardb_x_clinvar/splicevardb_x_clinvar_snv_chrX.vcf.gz
Indexing ./data/splicevardb_x_clinvar/splicevardb_x_clinvar_snv_chrX.atomized.vcf.gz


Lines   total/split/joined/realigned/removed/skipped:	511/0/0/0/0/0
REF/ALT total/modified/added:  	511/0/0


,CHROM,start,end
0,chrX,640852,640853
1,chrX,3615805,3615806
2,chrX,3615831,3615832
3,chrX,3615856,3615857
4,chrX,3615864,3615865


Saved padded BED to ./data/splicevardb_x_clinvar/splicevardb_x_clinvar_snv_chrX_region_bed.bed


In [42]:
# Split 5' and 3' ss annotation files based on chromosome

split_annotation_file = True
anno_dir = './data/annotation/'

if split_annotation_file:
    import pandas as pd
    p5_bed_file = pd.read_csv('./data/annotation/p5_bed.bed', sep='\t')
    p3_bed_file = pd.read_csv('./data/annotation/p3_bed.bed', sep='\t')
    
    chromosomes = [f"chr{i}" for i in range(1, 23)] + ["chrX"]
    
    # Loop through each chromosome
    for chrom in chromosomes:
        print(f"\n🔄 Splitting annotation file for chromosome: {chrom}...")
        chr_p5 = p5_bed_file[p5_bed_file['#CHROM'] == chrom]
        chr_p3 = p3_bed_file[p3_bed_file['#CHROM'] == chrom]
        chr_p5.to_csv(os.path.join(anno_dir, f"p5_{chrom}_bed.bed"), index=False, sep='\t', header=False)
        chr_p3.to_csv(os.path.join(anno_dir, f"p3_{chrom}_bed.bed"), index=False, sep='\t', header=False)


🔄 Splitting annotation file for chromosome: chr1...

🔄 Splitting annotation file for chromosome: chr2...

🔄 Splitting annotation file for chromosome: chr3...

🔄 Splitting annotation file for chromosome: chr4...

🔄 Splitting annotation file for chromosome: chr5...

🔄 Splitting annotation file for chromosome: chr6...

🔄 Splitting annotation file for chromosome: chr7...

🔄 Splitting annotation file for chromosome: chr8...

🔄 Splitting annotation file for chromosome: chr9...

🔄 Splitting annotation file for chromosome: chr10...

🔄 Splitting annotation file for chromosome: chr11...

🔄 Splitting annotation file for chromosome: chr12...

🔄 Splitting annotation file for chromosome: chr13...

🔄 Splitting annotation file for chromosome: chr14...

🔄 Splitting annotation file for chromosome: chr15...

🔄 Splitting annotation file for chromosome: chr16...

🔄 Splitting annotation file for chromosome: chr17...

🔄 Splitting annotation file for chromosome: chr18...

🔄 Splitting annotation file for chro

In [43]:
for name, path in vcf_paths.items():
    vcf_in = pysam.VariantFile(path, "r")
    print(list(vcf_in.header.contigs))

['1']
['2']
['3']
['4']
['5']
['6']
['7']
['8']
['9']
['10']
['11']
['12']
['13']
['14']
['15']
['16']
['17']
['18']
['19']
['20']
['21']
['22']
['X']


In [15]:
import os
from pathlib import Path
from sys import path
from src.dataset import SpliceHapDataset

# Define paths based on your previous pipeline
data_dir = Path("./data/gvl")
data_dir.mkdir(parents=True, exist_ok=True)

# Define input files (assuming chr22 based on example)
reference_bgz = Path(f"{reference_path}.bgz")
region_bed_path = Path("./data/splicevardb_x_clinvar/splicevardb_x_clinvar_snv_chr22_region_bed.bed")
annotation_bed_path = [
    ("3pss", "./data/annotation/p3_chr22_bed.bed"),
    ("5pss", "./data/annotation/p5_chr22_bed.bed")
]
variant_path = Path("./data/splicevardb_x_clinvar/splicevardb_x_clinvar_snv_chr22.atomized.vcf.gz")  # Curate splicing snvs
hap_pgen_path = Path("./data/1000_Genomes_on_GRCh38/vcf/ALL.chr22.shapeit2_integrated_snvindels_v2a_27022019.GRCh38.phased.vcf.atomized.vcf.gz")  # 1KGP haplotype variants
dataset_path = data_dir / "chr22_dataset.gvl"

# Construct the dataset
from src.dataset import SpliceHapDataset  # Assuming this is where the class is located

dataset = SpliceHapDataset.build_from_files_with_matching_sites(
    reference_path=str(reference_bgz),
    region_bed_path=str(region_bed_path),
    hap_pgen_path=str(hap_pgen_path),
    variant_path=str(variant_path),
    window_size=50,
    context_size=5000,
    bed_paths=annotation_bed_path,
    dataset_path=str(dataset_path),
    ploidy=2,
    deduplicate=False,
    remake_dataset=True,  # Set True for initial run
    num_workers=0,
    enable_profiling=True,
    enable_cache=True
)

# Output the shape of the dataset
dataset_shape = dataset.shape()
dataset_shape


2025-05-30 15:26:43.195 | INFO     | genvarloader._dataset._write:write:75 - Writing dataset to data/gvl/chr22_dataset.gvl
2025-05-30 15:26:43.196 | INFO     | genvarloader._dataset._write:write:82 - Found existing GVL store, overwriting.
2025-05-30 15:26:43.246 | INFO     | genvarloader._dataset._write:write:147 - Using 2548 samples.
2025-05-30 15:26:43.247 | INFO     | genvarloader._dataset._write:write:153 - Writing genotypes.
2025-05-30 15:26:43.252 | INFO     | genvarloader._dataset._write:_write_from_vcf:230 - VCF genoray index is invalid, writing
Reading records: 100%|████████████████████████████████████████| 1059079/1059079 [00:16<00:00, 63736.46 record/s]
2025-05-30 15:27:10.513 | INFO     | genoray._vcf:_load_index:1028 - Loading genoray index.
Processing genotypes for 123 regions on contig chr22: 100%|██████████████| 123/123 [00:30<00:00,  4.05 region/s]
2025-05-30 15:27:42.953 | INFO     | genvarloader._dataset._write:write:177 - Finished writing.
2025-05-30 15:27:42.960 | 

(123, 2548, 2)

In [16]:
# reference = '/grid/kinney/home/zhliu/.cache/pooch/296ca04ba1df562072adc4c76f64cfb9-GRCh38_full_analysis_set_plus_decoy_hla.fa.bgz'
# region_bed_path = '../data/splicing/splicevardb_x_clinvar_snv_ch22_region_bed.bed'
# atomized_variants = '/grid/kinney/home/zhliu/.cache/pooch/ALL.chr22.shapeit2_integrated_snvindels_v2a_27022019.GRCh38.phased.renamed.atomized.vcf.gz'
# snps = '../data/splicing/splicevardb_x_clinvar_snv_ch22.atomized.vcf.gz'
# annotation_bed_path = [('3pss', '../data/p3_chr22_bed.bed'), ('5pss', '../data/p5_chr22_bed.bed')]

# dataset1 = SpliceHapDataset.build_from_files_with_matching_sites(
#     reference_path=reference,
#     region_bed_path=region_bed_path,
#     hap_pgen_path=atomized_variants,
#     variant_path=snps,
#     window_size=50,
#     context_size=5000,
#     bed_paths=annotation_bed_path,
#     dataset_path='./test_full_chr22_dataset_var_bed.gvl',
#     ploidy=2,
#     deduplicate=False,
#     remake_dataset=False, #this needs to be set to true the first time the dataset is built but takes a bit so should be set to False in future loads (unless you have changed a specification file)
#     num_workers=0, #adding workers to the dataset this makes batched loading faster (but has not been extensively tested) can be set to 0 for no workers
#     enable_profiling=False, # this should be set to true if you want to analyze the performance of the dataset. should be false for normal use
#     enable_cache=True, #If this is set to True the loading of sequential variants will be wildly faster but the initial loading will be slower. If you are going to be loading non-sequential variants then set this to False.
#     )


# dataset1.shape()

2025-05-30 15:33:15.194 | INFO     | genvarloader._dataset._impl:open:192 - Loading reference genome into memory. This typically has a modest memory footprint (a few GB) and greatly improves performance.
2025-05-30 15:33:15.344 | INFO     | genvarloader._dataset._reconstruct:from_path:183 - Loading variant data.
2025-05-30 15:33:15.400 | INFO     | genvarloader._dataset._impl:open:278 - Opened dataset:
GVL store at test_full_chr22_dataset_var_bed.gvl
Is subset: False
# of regions: 123
# of samples: 2548
Output length: ragged
Jitter: 0 (max: 0)
Deterministic: True
Sequence type: reference [haplotypes] annotated variants
Active tracks: None
Tracks available: None

Reading records: 100%|███████████████████████████████████████████████| 123/123 [00:00<00:00, 226669.33 record/s]
/grid/kinney/home/zhliu/vep_dna/VEP_splicing/testing/src/dataset.py:239: DeprecationWarning: `DataFrame.with_row_count` is deprecated; use `with_row_index` instead. Note that the default column name has changed from 

(123, 2548, 2)

In [28]:
# # Test loading on a batch of variants
# dataset.enable_profiling = False
# dataset.enable_cache = True
# dataset_output =next(dataset.batch_iter(2000))

# # Test loading on a batch of variants
# dataset1.enable_profiling = False
# dataset1.enable_cache = True
# dataset1_output =next(dataset1.batch_iter(2000))

In [30]:
# for idx in range(len(dataset_output)):
#     print((dataset1_output[idx] == dataset2_output[idx]).all())

True
True
True
True
True
True
True
True
True
True
True


In [44]:
import os
from pathlib import Path
from src.dataset import SpliceHapDataset

# Define root paths
data_dir = Path("./data/gvl")
data_dir.mkdir(parents=True, exist_ok=True)
reference_bgz = Path(f"{reference_path}.bgz")  # From earlier step

# List of chromosomes to process
chromosomes = [f"chr{i}" for i in range(1, 23)] + ["chrX"]

# Loop through each chromosome
for chrom in chromosomes:
    print(f"\n🔄 Processing {chrom}...")

    variant_path = Path(f"./data/splicevardb_x_clinvar/splicevardb_x_clinvar_snv_{chrom}.atomized.vcf.gz")
    hap_pgen_path = Path(f"./data/1000_Genomes_on_GRCh38/vcf/ALL.{chrom}.shapeit2_integrated_snvindels_v2a_27022019.GRCh38.phased.vcf.atomized.vcf.gz")
    region_bed_path = Path(f"./data/splicevardb_x_clinvar/splicevardb_x_clinvar_snv_{chrom}_region_bed.bed")
    annotation_bed_path = [
        ("3pss", f"./data/annotation/p3_{chrom}_bed.bed"),
        ("5pss", f"./data/annotation/p5_{chrom}_bed.bed")
    ]
    dataset_path = data_dir / f"{chrom}_dataset.gvl"

    # ⏭️ Skip if output already exists
    if dataset_path.exists():
        print(f"⏭️ {chrom} → Output already exists, skipping...")
        continue

    # Check if all required files exist
    required_files = [variant_path, hap_pgen_path, region_bed_path] + [Path(p[1]) for p in annotation_bed_path]
    if not all(f.exists() for f in required_files):
        print(f"⚠️ Missing files for {chrom}, skipping...")
        continue

    # Build the dataset
    dataset = SpliceHapDataset.build_from_files_with_matching_sites(
        reference_path=str(reference_bgz),
        region_bed_path=str(region_bed_path),
        hap_pgen_path=str(hap_pgen_path),
        variant_path=str(variant_path),
        window_size=50,
        context_size=5000,
        bed_paths=annotation_bed_path,
        dataset_path=str(dataset_path),
        ploidy=2,
        deduplicate=False,
        remake_dataset=True,  # Set False in future to avoid rebuilding
        num_workers=0,
        enable_profiling=True,
        enable_cache=True
    )

    shape = dataset.shape()
    print(f"✅ {chrom} → Dataset shape: {shape}")



🔄 Processing chr1...


2025-05-30 16:20:18.793 | INFO     | genvarloader._dataset._write:write:75 - Writing dataset to data/gvl/chr1_dataset.gvl
2025-05-30 16:20:18.872 | INFO     | genvarloader._dataset._write:write:147 - Using 2548 samples.
2025-05-30 16:20:18.873 | INFO     | genvarloader._dataset._write:write:153 - Writing genotypes.
2025-05-30 16:20:18.879 | INFO     | genvarloader._dataset._write:_write_from_vcf:230 - VCF genoray index is invalid, writing
Reading records: 100%|████████████████████████████████████████| 6191833/6191833 [01:27<00:00, 70613.15 record/s]
2025-05-30 16:22:38.457 | INFO     | genoray._vcf:_load_index:1028 - Loading genoray index.
Processing genotypes for 560 regions on contig chr1: 100%|███████████████| 560/560 [02:15<00:00,  4.13 region/s]
2025-05-30 16:25:01.518 | INFO     | genvarloader._dataset._write:write:177 - Finished writing.
2025-05-30 16:25:01.523 | INFO     | genvarloader._dataset._impl:open:192 - Loading reference genome into memory. This typically has a modest m

✅ chr1 → Dataset shape: (560, 2548, 2)

🔄 Processing chr2...


Reading records: 100%|████████████████████████████████████████| 6790551/6790551 [01:34<00:00, 71684.81 record/s]
2025-05-30 16:27:35.829 | INFO     | genoray._vcf:_load_index:1028 - Loading genoray index.
Processing genotypes for 775 regions on contig chr2: 100%|███████████████| 775/775 [03:09<00:00,  4.09 region/s]
2025-05-30 16:30:53.732 | INFO     | genvarloader._dataset._write:write:177 - Finished writing.
2025-05-30 16:30:53.736 | INFO     | genvarloader._dataset._impl:open:192 - Loading reference genome into memory. This typically has a modest memory footprint (a few GB) and greatly improves performance.
2025-05-30 16:30:54.323 | INFO     | genvarloader._dataset._reconstruct:from_path:183 - Loading variant data.
2025-05-30 16:30:54.740 | INFO     | genvarloader._dataset._impl:open:278 - Opened dataset:
GVL store at data/gvl/chr2_dataset.gvl
Is subset: False
# of regions: 775
# of samples: 2548
Output length: ragged
Jitter: 0 (max: 0)
Deterministic: True
Sequence type: reference [

✅ chr2 → Dataset shape: (775, 2548, 2)

🔄 Processing chr3...


...

Processing genotypes for 123 regions on contig chr22: 100%|██████████████| 123/123 [00:29<00:00,  4.20 region/s]
2025-05-30 17:24:21.195 | INFO     | genvarloader._dataset._write:write:177 - Finished writing.
2025-05-30 17:24:21.199 | INFO     | genvarloader._dataset._impl:open:192 - Loading reference genome into memory. This typically has a modest memory footprint (a few GB) and greatly improves performance.
2025-05-30 17:24:21.252 | INFO     | genvarloader._dataset._reconstruct:from_path:183 - Loading variant data.
2025-05-30 17:24:21.332 | INFO     | genvarloader._dataset._impl:open:278 - Opened dataset:
GVL store at data/gvl/chr22_dataset.gvl
Is subset: False
# of regions: 123
# of samples: 2548
Output length: ragged
Jitter: 0 (max: 0)
Deterministic: True
Sequence type: reference [haplotypes] annotated variants
Active tracks: None
Tracks available: None

Reading records: 100%|███████████████████████████████████████████████| 123/123 [00:00<00:00, 261082.69 record/s]
/grid/kinney/hom

✅ chr22 → Dataset shape: (123, 2548, 2)

🔄 Processing chrX...


Reading records: 100%|██████████████████████████████████████████| 106963/106963 [00:01<00:00, 65933.28 record/s]
2025-05-30 17:24:24.047 | INFO     | genoray._vcf:_load_index:1028 - Loading genoray index.
Processing genotypes for 511 regions on contig chrX:   0%|                 | 1/511 [00:00<02:51,  2.97 region/s]2025-05-30 17:24:24.492 | WARNING  | genvarloader._dataset._write:_write_from_vcf:287 - A region has no variants for any sample. This could be expected depending on the region lengths and source of variants. However, this can also be caused by a mismatch between the reference genome used for the BED file coordinates and the one used for the variants. This warning will not be shown again.
Processing genotypes for 511 regions on contig chrX: 100%|██████████████| 511/511 [00:03<00:00, 167.87 region/s]
2025-05-30 17:24:27.584 | INFO     | genvarloader._dataset._write:write:177 - Finished writing.
2025-05-30 17:24:27.588 | INFO     | genvarloader._dataset._impl:open:192 - Loading

✅ chrX → Dataset shape: (511, 2548, 2)



/grid/kinney/home/zhliu/vep_dna/VEP_splicing/testing/src/dataset.py:239: DeprecationWarning: `DataFrame.with_row_count` is deprecated; use `with_row_index` instead. Note that the default column name has changed from 'row_nr' to 'index'.
  self.matching_indices = self.gvl.rows.with_row_count().filter(pl.col("region_idx") == pl.col("site_idx")).get_column("row_nr").to_numpy()
